# 00 — hzz raw 预处理 sanity check

`scripts/preprocess_hzz_raw.py` 是新增的管线阶段：把 `data/hzz_raw/` 下 12 个 day + 4 个 mon 原始 CSV → 8 个 per-table 文件（4 day + 4 mon）→ 可选 2 个 horizontal-merged 文件。

本 notebook 不重跑预处理，只读产物，回答四件事：
1. **header alias 命中** —— 4 个 table 的时间列分别叫什么？是否都被改名为 `cd_time`？
2. **shard 均匀度** —— `shard_count` 个 id-hash shard 的行数分布平不平？歪了说明 id 分布有偏，dedup/merge 内存峰值会失控。
3. **dedup 折叠比** —— 各 table 期望的「~5% 重复行」有没有真的被合掉？
4. **horizontal merge 命中率** —— 4 个 per-table 在 (cust_id, cd_time) 上的交集占比，决定 outer/inner/left 哪种 join 不会丢数据。

默认配置切到 **`hzz_raw_synth`**（合成数据，跑得快）；要审计真实数据，把 `CFG_NAME` 改成 `'hzz_raw'` 即可。

In [ ]:
import sys
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '../src')
sys.path.insert(0, '../scripts')                              # 复用预处理脚本里的 shard_of / canonical_header
from preprocess_hzz_raw import shard_of, canonical_header     # noqa: E402

REPO = Path('..').resolve()
CFG_NAME = 'hzz_raw_synth'                                    # 'hzz_raw' = 真实数据
cfg = yaml.safe_load((REPO / 'configs/preprocess' / f'{CFG_NAME}.yaml').read_text())
INPUT_DIR  = REPO / cfg['input_dir']
OUTPUT_DIR = REPO / cfg['output_dir']
ID_COL    = cfg['id_col']
TIME_COL  = cfg['time_col']
ALIASES   = cfg['time_col_aliases']
SHARDS    = int(cfg.get('shard_count', 8))
print('input :', INPUT_DIR)
print('output:', OUTPUT_DIR)
print('shards:', SHARDS)
if not INPUT_DIR.is_dir():
    print(f'\n⚠ {INPUT_DIR} 不存在 —— 跑 `python3 scripts/build_hzz_raw_synthetic.py` 先生成合成数据。')
if not OUTPUT_DIR.is_dir():
    print(f'⚠ {OUTPUT_DIR} 不存在 —— 跑 `python3 scripts/preprocess_hzz_raw.py --config configs/preprocess/{CFG_NAME}.yaml --merge` 先生成 per-table + merged。')

## 1. Header alias 命中

生产里 4 个 feature_table 的时间列名互相不一致（`cd_time` / `dt` / `statis_date` / …），预处理脚本统一改名到 `cd_time`。这里把每个原始文件的 alias 命中情况列出来——若同一 table 不同 row chunk 命中不同 alias，预处理已会 fail；这里只是肉眼复核。

In [ ]:
rows = []
for gran, src_cfg in cfg['sources'].items():
    n_rows = int(src_cfg['row_chunks'])
    n_tables = int(src_cfg['feature_tables'])
    pat = src_cfg['file_pattern']
    for table in range(1, n_tables + 1):
        if n_rows == 1:
            paths = [INPUT_DIR / gran / pat.format(table=table)]
        else:
            paths = [INPUT_DIR / gran / pat.format(row=r, table=table)
                     for r in range(1, n_rows + 1)]
        for p in paths:
            if not p.is_file():
                rows.append({'gran': gran, 'table': table, 'file': p.name,
                              'alias': '<MISSING>', 'n_cols': None}); continue
            head = pd.read_csv(p, nrows=0).columns
            hits = [a for a in ALIASES if a in head]
            rows.append({'gran': gran, 'table': table, 'file': p.name,
                          'alias': hits[0] if hits else '<NONE>',
                          'n_cols': len(head)})
header_df = pd.DataFrame(rows)
print(header_df.to_string(index=False))

mismatches = header_df.groupby(['gran', 'table'])['alias'].nunique()
bad = mismatches[mismatches > 1]
if len(bad):
    print('\n⚠ 同一 table 不同 row chunk 命中了不同 alias：')
    print(bad)
else:
    print('\n✓ 每个 table 的 row chunks 都命中同一 alias。')

## 2. Shard 均匀度

`preprocess_hzz_raw.py` 在 Stage B (per-table 聚合) 和 Stage C (horizontal merge) 都按 `hash(cust_id) % shard_count` 分片，shard 之间独立处理；歪斜的 shard 决定内存峰值。读 per-table 输出统计一下。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, gran in zip(axes, ['day', 'mon']):
    src_cfg = cfg['sources'][gran]
    n_tables = int(src_cfg['feature_tables'])
    bottoms = np.zeros(SHARDS)
    for table in range(1, n_tables + 1):
        out_path = OUTPUT_DIR / src_cfg['per_table_out'].format(table=table)
        if not out_path.is_file():
            print(f'skip {out_path} (missing)'); continue
        ids = pd.read_csv(out_path, usecols=[ID_COL])[ID_COL]
        s = shard_of(ids, SHARDS)
        counts = pd.Series(s).value_counts().reindex(range(SHARDS), fill_value=0)
        ax.bar(range(SHARDS), counts.values, bottom=bottoms,
                label=f't{table}', alpha=0.85)
        bottoms += counts.values
    ax.set_xlabel('shard'); ax.set_ylabel('rows')
    ax.set_title(f'{gran} — rows per shard (stacked by table)')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('歪斜判断：max/mean > ~1.5 说明 id 分布偏，建议把 shard_count 调小或换 hash 列。')

## 3. Dedup 折叠比

`dedup.ignore_time=True` 下，「同 id 同特征值不同 cd_time」的行会被合并。比对 *输入文件总行数 → 输出文件行数*，确认折叠确实发生且幅度合理。

合成数据里 `--dup-frac=0.05` 注入了 5% 重复，预期看到 ≥5% 的折叠比。

In [ ]:
rows = []
for gran, src_cfg in cfg['sources'].items():
    n_rows_chunks = int(src_cfg['row_chunks'])
    n_tables = int(src_cfg['feature_tables'])
    pat = src_cfg['file_pattern']
    for table in range(1, n_tables + 1):
        if n_rows_chunks == 1:
            in_paths = [INPUT_DIR / gran / pat.format(table=table)]
        else:
            in_paths = [INPUT_DIR / gran / pat.format(row=r, table=table)
                        for r in range(1, n_rows_chunks + 1)]
        # 行数：sum(input) vs output —— 不强行 read 全表，只读 1 列以省内存
        in_n = sum(len(pd.read_csv(p, usecols=[ID_COL])) for p in in_paths if p.is_file())
        out_path = OUTPUT_DIR / src_cfg['per_table_out'].format(table=table)
        out_n = len(pd.read_csv(out_path, usecols=[ID_COL])) if out_path.is_file() else None
        ratio = (1 - out_n / in_n) if (out_n is not None and in_n) else None
        rows.append({
            'gran': gran, 'table': table,
            'rows_in': in_n, 'rows_out': out_n,
            'dedup_ratio': f'{ratio:.2%}' if ratio is not None else '?',
        })
dedup_df = pd.DataFrame(rows)
print(dedup_df.to_string(index=False))

## 4. Horizontal merge 命中率

Stage C 把 4 个 per-table 在 `(cust_id, cd_time)` 上 outer-merge 成一张宽表。要看：
- merged 行数 vs 各 per-table 行数（outer 应当 ≥ max(per-table)）
- 每个 per-table 在 merged 中的命中率（merged join key 出现在该 table 的比例）—— 命中率低的 table 在 outer 下会贡献大量 NaN，inner 下会成为吞吐瓶颈。

若没跑 merge（`merge.enabled=false` 或没传 `--merge`），跳过。

In [ ]:
for gran, src_cfg in cfg['sources'].items():
    merged_path = OUTPUT_DIR / src_cfg['merged_out']
    if not merged_path.is_file():
        print(f'{gran}: 没找到 {merged_path} —— 加 --merge 重跑预处理或在 yaml 里 enable.')
        continue
    keys_merged = pd.read_csv(merged_path, usecols=[ID_COL, TIME_COL]).drop_duplicates()
    print(f'\n--- {gran} merged ---  rows={len(keys_merged)}')
    n_tables = int(src_cfg['feature_tables'])
    for table in range(1, n_tables + 1):
        out_path = OUTPUT_DIR / src_cfg['per_table_out'].format(table=table)
        if not out_path.is_file():
            continue
        keys_t = pd.read_csv(out_path, usecols=[ID_COL, TIME_COL]).drop_duplicates()
        joined = keys_merged.merge(keys_t.assign(_in=1), on=[ID_COL, TIME_COL], how='left')
        hit = joined['_in'].notna().mean()
        print(f'  t{table}: per-table rows={len(keys_t)}  hit-in-merged={hit:.2%}'
              f'  → 该 table 给 {hit:.0%} 行贡献了非 NaN 值')

## 结论 / 用法

- §1 的目的是兜底：如果生产某天某个 table 的时间列名突然变了（`cd_time` → `partition_date`），加 alias 到 `time_col_aliases` 即可，不用动代码。
- §2 的歪斜阈值 ~1.5：超了说明 hash 在你的 id 分布上不够均匀。补救最简单：把 `shard_count` 翻倍（每个 shard 内存占用减半）。
- §3 的折叠比应当跟生产环境的「全量 vs 增量」节奏吻合。0% 折叠（dedup 没起作用）是 bug；50%+ 折叠说明上游 dump 有问题或者 dedup 配置失误。
- §4 的命中率告诉你 merge 策略选什么：`hit ≥ 0.95` 全部 inner 也行；某 table `hit < 0.5` 必须 outer + 让模型容忍大量 NaN。